# 📦 MeatGuard-S — Distillation, Compression & Export

**Phase 1 · RO2** — Teacher → Student knowledge distillation, L1 structured pruning + recovery fine-tune, then multi-format export (**`.pth` → `.onnx` → `.tflite`**).

Mirrors `src/model/kd_trainer.py`, `src/model/distillation.py`, `src/model/compression.py`, `src/utils/export.py`.

## 1. Setup

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Run once on a fresh Colab/Kaggle runtime (uncomment):
# !pip -q install torch torchvision timm einops scikit-learn matplotlib seaborn tqdm opencv-python shap scikit-image onnx tensorflow onnx2tf onnxscript

In [ ]:
import os, sys

# --- Path bootstrap: make ../src importable on local, Colab, and Kaggle ------
def add_src_to_path():
    here = os.getcwd()
    candidates = [
        os.path.abspath(os.path.join(here, "..", "src")),   # local: notebooks/ -> ../src
        os.path.abspath(os.path.join(here, "src")),         # run from repo root
        "/content/drive/MyDrive/MeatGuard_II",                        # Colab (git clone target)
        "/kaggle/working/MeatGuard_II/src",                 # Kaggle
    ]
    for c in candidates:
        if os.path.isdir(c):
            if c not in sys.path:
                sys.path.insert(0, c)
            return c
    raise FileNotFoundError(
        "Could not locate the MeatGuard_II/src package. "
        "On Colab/Kaggle, clone the repo into /content (or /kaggle/working) first."
    )

SRC_DIR = add_src_to_path()
print("Using src package at:", SRC_DIR)
os.chdir(SRC_DIR)
print(f"Changed working directory to: {os.getcwd()}")

In [ ]:
import torch
import onnxscript
import torch.nn as nn
import tensorflow as tf
from onnx2tf import convert
from src.data.dataset import make_loaders
from src.model.architecture import MeatGuardS
from src.model.kd_trainer import KDTrainer
from src.model.compression import MeatGuardPruner
from src.utils.export import export_to_tflite

In [ ]:
# Placeholder dataset (drop-in stand-in for the full MeatScan set).
# Override with:  os.environ["MEATGUARD_DATA"] = "/path/to/your/data"
DATA_DIR = os.environ.get("MEATGUARD_DATA", r"/content/drive/MyDrive/Datasets/MeatScan/test")
ARTIFACTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "artifacts"))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("Data dir :", DATA_DIR)
print("Artifacts:", ARTIFACTS_DIR)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader, val_loader, test_loader, _ = make_loaders(DATA_DIR, strategy="domain")

## 2. Knowledge distillation

The **teacher** is selected by `00_teacher_training.ipynb`: an ImageNet-pretrained model fine-tuned on the
**session split of the cropped images** and chosen on validation macro-F1. (The Project-1 MobileViT checkpoint is
*not* used: it has only 0.16 M parameters, fewer than MeatGuard-S, and was trained on a random image-level split, so
it has seen the current test sessions.) `KDTrainer` rebuilds the teacher from its checkpoint and combines soft-label
KD (T = 4, α = 0.3) with alignment of the student's Stage-3 features to the teacher's penultimate features.

In [ ]:
import json
# teacher_selection.json is written by `python -m src.train_teacher select` (notebook 00)
TEACHER_RUNS = os.environ.get("MEATGUARD_TEACHER_RUNS", os.path.join(ARTIFACTS_DIR, "..", "runs", "teacher"))
selection = json.load(open(os.path.join(TEACHER_RUNS, "teacher_selection.json")))
TEACHER_CKPT = selection["checkpoint"]
print("Teacher:", selection["selected_teacher"], "->", TEACHER_CKPT)
print("Teacher test macro-F1:", round(selection["test"]["macro_f1"], 4))
teacher = TEACHER_CKPT   # KDTrainer accepts the checkpoint path and loads the model itself

In [ ]:
student = MeatGuardS(num_classes=2)
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)

if teacher is not None:
    kd_trainer = KDTrainer(student, teacher, train_loader, val_loader, optimizer, device=device)
    for epoch in range(50):
        loss = kd_trainer.train_epoch(beta=0.1)
        print(f"KD epoch {epoch+1:02d} | loss {loss:.4f}")
else:
    print("No teacher available — load a pretrained student or run notebook 01 first.")

## 3. Compression — L1 structured pruning + recovery fine-tune

Prunes ~30% of channels in 3×3 conv layers, then briefly fine-tunes to recover accuracy.

In [ ]:
pruner = MeatGuardPruner(student)
pruner.prune_conv_layers(amount=0.3)
pruner.recover_fine_tune(
    train_loader, val_loader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=torch.optim.AdamW(student.parameters(), lr=1e-4),
    device=device, epochs=10,
)

## 4. Multi-format export — `.pth` / `.onnx` / `.tflite`

PyTorch is the source of truth; ONNX is the portable IR; TFLite (INT8) is the on-device runtime. The TFLite step needs TensorFlow + onnx-tf in the environment and is guarded.

In [ ]:
student.eval().cpu()
pth_path  = os.path.join(ARTIFACTS_DIR, "meatguard_s_final.pth")
onnx_path = os.path.join(ARTIFACTS_DIR, "meatguard_s.onnx")
saved_model_dir = os.path.join(
    ARTIFACTS_DIR, "saved_model"
)

tflite_path = os.path.join(
    ARTIFACTS_DIR, "meatguard_s_int8.tflite"
)

# .pth
# The pruned student no longer matches the MeatGuardS constructor, so save the whole module
torch.save(student, pth_path)   # reload with torch.load(pth_path, weights_only=False)
print("Saved", pth_path)

# .onnx
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    student, dummy, onnx_path,
    opset_version=18, # Changed opset_version from 13 to 18
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    dynamo=False # Explicitly disable TorchDynamo for ONNX export
)
print("Saved", onnx_path)

In [ ]:
# ONNX -> TensorFlow SavedModel
convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=saved_model_dir,
)

print("Saved TensorFlow model:", saved_model_dir)

In [ ]:
# .tflite (INT8). Calibrates on the train loader. Requires TF + onnx-tf.
tflite_path = os.path.join(ARTIFACTS_DIR, "meatguard_s_int8.tflite")
try:
    export_to_tflite(student, train_loader, model_path=tflite_path)
except Exception as e:
    print(f"[tflite] export skipped in this environment: {e}")

> **Parity check (recommended):** run the same calibration images through the PyTorch, ONNX, and TFLite models and assert the logits/argmax agree within tolerance before shipping the `.tflite` to the mobile app.